# Mission 03 — Multi-Head Attention, RoPE Vector Rotation & Grouped-Query KV-Cache Optimization

**Starship Operational Unit:** USSE Noosphere Bridge Crew  
**Sector:** `SEC-03-M45` (Pleiades Open Cluster, Seven Sisters)  
**Assigned Officer:** Syed Hashir Sohail — Systems Architect / Lead Engineer  
**Squad:** Sutskever (Group 18)  
**Deliverable:** Part 1 Core Architecture & Verification Notebook (`ps03_solution.ipynb`)

---

### Notebook Architecture & Contents

This notebook drives and validates the core self-attention layer implemented in `01_MultiHead_Attention_Core_starter.py`:

| Section | Topic | What It Demonstrates |
|:---|:---|:---|
| **0** | **Setup & Environment** | Imports, PyTorch environment configuration, seed fixing |
| **1** | **Multi-Head Self-Attention** | Pure PyTorch tensor projections, head splitting, and forward pass |
| **2** | **Variance Preservation** | Empirical validation of $1/\sqrt{d_k}$ scaling preventing vanishing/exploding gradients |
| **3** | **Causal Masking** | Lower-triangular masking preserving autoregressive causality |
| **4** | **Padding & Stability** | Boolean padding masks, fully masked row handling, FP16/BF16 stability |
| **5** | **Reference Verification** | Independent unit tests comparing against PyTorch SDPA test oracle |
| **6** | **Compatibility Check (Part 3)** | Cross-module equivalence between MHA and GQA ($H_Q = H_{KV}$) |
| **7** | **Review & Sign-Off** | Status summary and audit sign-off |


## 0. Setup and Environment

Verify PyTorch version and set fixed random seeds for strict determinism.


In [1]:
import importlib.util
import math
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
device = torch.device("cpu")
torch.manual_seed(42)

HERE = Path.cwd()
if not (HERE / "01_MultiHead_Attention_Core_starter.py").is_file():
    PART1_PATH = HERE / "mission-03" / "01_MultiHead_Attention_Core_starter.py"
    PART3_PATH = HERE / "mission-03" / "03_GQA_and_KVCache_Benchmark_starter.py"
else:
    PART1_PATH = HERE / "01_MultiHead_Attention_Core_starter.py"
    PART3_PATH = HERE / "03_GQA_and_KVCache_Benchmark_starter.py"

spec1 = importlib.util.spec_from_file_location("mha_core", PART1_PATH)
mha_core = importlib.util.module_from_spec(spec1)
spec1.loader.exec_module(mha_core)

MultiHeadSelfAttention = mha_core.MultiHeadSelfAttention
print("MultiHeadSelfAttention loaded successfully from:", PART1_PATH.name)


PyTorch Version: 2.14.1+cpu
CUDA Available: False
MultiHeadSelfAttention loaded successfully from: 01_MultiHead_Attention_Core_starter.py


## 1. Multi-Head Self-Attention Architecture

The `MultiHeadSelfAttention` module implements:
1. **Linear Projections**: $W_Q, W_K, W_V \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$ and output projection $W_O$.
2. **Head Splitting**: Reshapes $(B, T, d_{\text{model}}) \to (B, h, T, d_k)$ where $d_k = d_{\text{model}} / h$.
3. **Scaled Dot-Product Attention**: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$.


In [2]:
batch_size = 2
seq_len = 8
d_model = 64
num_heads = 4
head_dim = d_model // num_heads

mha = MultiHeadSelfAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)
mha.eval()

x = torch.randn(batch_size, seq_len, d_model)
output, weights = mha(x, causal=False)

print(f"Input shape:             {list(x.shape)}")
print(f"Output shape:            {list(output.shape)}")
print(f"Attention weights shape: {list(weights.shape)} (Batch, Heads, Seq_Q, Seq_K)")
print(f"Weights sum across keys: {weights.sum(dim=-1)[0, 0].detach().numpy()} (All equal 1.0)")


Input shape:             [2, 8, 64]
Output shape:            [2, 8, 64]
Attention weights shape: [2, 4, 8, 8] (Batch, Heads, Seq_Q, Seq_K)
Weights sum across keys: [1.         0.99999994 1.0000001  1.         1.         1.
 1.         0.99999994] (All equal 1.0)


## 2. Mathematical Variance Preservation ($\frac{1}{\sqrt{d_k}}$)

### Theoretical Foundation:
If components of $q \in \mathbb{R}^{d_k}$ and $k \in \mathbb{R}^{d_k}$ are independent identically distributed random variables with $\mathbb{E}[q_i] = \mathbb{E}[k_i] = 0$ and $\text{Var}(q_i) = \text{Var}(k_i) = 1$:
$$\mathbb{E}[q \cdot k] = \sum_{i=1}^{d_k} \mathbb{E}[q_i k_i] = 0$$
$$\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = \sum_{i=1}^{d_k} \mathbb{E}[q_i^2] \mathbb{E}[k_i^2] = d_k$$
Without scaling, for large $d_k$ (e.g. 64 or 128), $\text{Var}(q \cdot k) = d_k$. As a result, the dot products grow large in magnitude, pushing softmax into saturation with vanishing gradients.
Scaling by $\frac{1}{\sqrt{d_k}}$ ensures:
$$\text{Var}\left(\frac{q \cdot k}{\sqrt{d_k}}\right) = \frac{1}{d_k} \text{Var}(q \cdot k) = \frac{d_k}{d_k} = 1.0$$


In [3]:
print("Running verify_variance_preservation() from 01_MultiHead_Attention_Core_starter.py:\n")
mha_core.verify_variance_preservation()


Running verify_variance_preservation() from 01_MultiHead_Attention_Core_starter.py:

=== Subsystem 03: Attention Variance Preservation Verification ===
Head Dim (d_k): 64
Unscaled Dot Product — Mean: 0.0090, Variance: 64.1756 (Expected ~64)
Scaled Dot Product   — Mean: 0.0011, Variance: 1.0027 (Expected ~1.000)
[PASS] Variance preservation holds within tolerance! Subsystem 03 Attention Matrix stabilized.


## 3. Causal Attention Masking

In autoregressive generation, position $i$ must not attend to any future position $j > i$. Causal masking adds $-\infty$ to upper-triangular logits ($j > i$) so that their softmax probabilities become strictly 0.0.


In [4]:
out_causal, weights_causal = mha(x, causal=True)

print("Causal Attention Weight Matrix (Head 0, Batch 0):")
print(torch.round(weights_causal[0, 0] * 100) / 100)

upper_tri_nonzero = torch.count_nonzero(weights_causal.triu(1))
print(f"\nNon-zero elements in strictly upper triangle: {upper_tri_nonzero.item()}")
assert upper_tri_nonzero == 0, "Causal violation: non-zero attention in upper triangle!"
print("[PASS] Causal masking strictly preserves autoregressive structure.")


Causal Attention Weight Matrix (Head 0, Batch 0):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.6000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4100, 0.3200, 0.2800, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3400, 0.2800, 0.2300, 0.1400, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1600, 0.2600, 0.2300, 0.1800, 0.1700, 0.0000, 0.0000, 0.0000],
        [0.1400, 0.1800, 0.1400, 0.1300, 0.2100, 0.1900, 0.0000, 0.0000],
        [0.1400, 0.1500, 0.0900, 0.1600, 0.1300, 0.1500, 0.1900, 0.0000],
        [0.1600, 0.1200, 0.1300, 0.1400, 0.1000, 0.1500, 0.1200, 0.0900]],
       grad_fn=<DivBackward0>)

Non-zero elements in strictly upper triangle: 0
[PASS] Causal masking strictly preserves autoregressive structure.


## 4. Padding Masks, Numerical Stability (FP16/BF16) & Edge Cases

The implementation handles:
1. **Boolean padding masks**: Invalid key tokens receive 0 attention; masked query rows yield 0 outputs without NaN.
2. **Single-token sequence length**: $T=1$ passes without shape degeneration.
3. **Low-precision stability**: Accumulates attention scores in float32 for FP16 and BF16 tensors, avoiding intermediate overflow or underflow.


In [5]:
pad_mask = torch.tensor([
    [False, False, True, True, True, True, True, True],
    [True,  True,  True, True, True, True, True, True]
])

out_padded, weights_padded = mha(x, attention_mask=pad_mask, causal=True)
print("Padded sequence output is finite:", torch.isfinite(out_padded).all().item())
print("Padding key weights (columns 0-1) for batch 0:", weights_padded[0, 0, :, :2].max().item())
assert weights_padded[0, 0, :, :2].max() == 0.0, "Padding tokens received non-zero attention!"
print("[PASS] Padding keys are completely masked.")

for dtype in (torch.float16, torch.bfloat16):
    mha_low = MultiHeadSelfAttention(d_model=d_model, num_heads=num_heads, dropout=0.0).to(dtype).eval()
    x_low = x.to(dtype)
    out_low, _ = mha_low(x_low, causal=True)
    assert out_low.dtype == dtype and torch.isfinite(out_low).all()
    print(f"[PASS] {dtype} forward pass succeeds with finite outputs.")


Padded sequence output is finite:

 True
Padding key weights (columns 0-1) for batch 0: 0.0
[PASS] Padding keys are completely masked.
[PASS] torch.float16 forward pass succeeds with finite outputs.
[PASS] torch.bfloat16 forward pass succeeds with finite outputs.


## 5. Independent Verification against PyTorch SDPA Reference

We run the full validation suite `verify_attention_correctness()`, which verifies:
- Mathematical equivalence against `F.scaled_dot_product_attention`
- Output invariance to future tokens (causal isolation)
- Finite gradient backward pass with padding
- Validation on bad inputs and non-matching mask dimensions


In [6]:
print("Running verify_attention_correctness():\n")
mha_core.verify_attention_correctness()


Running verify_attention_correctness():



PASS: independent SDPA reference, row sums and causal isolation
PASS: left padding, fully masked rows and single-token attention
PASS: FP16 and BF16 forward passes and float32 agreement
PASS: finite backward gradients with padding and fully masked rows
PASS: mask shape and dtype validation


## 6. Compatibility Check with Part 3 (GQA & KV-Cache)

Part 3 implements Grouped-Query Attention (`GQAAttention`). When $H_Q = H_{KV}$ (i.e. every query head has its own key-value head), GQA mathematically degenerates to standard Multi-Head Attention.
Here we verify cross-module architectural agreement between Part 1 and Part 3.


In [7]:
spec3 = importlib.util.spec_from_file_location("gqa_core", PART3_PATH)
gqa_core = importlib.util.module_from_spec(spec3)
spec3.loader.exec_module(gqa_core)
GQAAttention = gqa_core.GQAAttention
KVCache = gqa_core.KVCache

print(f"Loaded {GQAAttention.__name__} and {KVCache.__name__} from Part 3.")

d_test = 64
heads_test = 4
torch.manual_seed(123)

mha_layer = MultiHeadSelfAttention(d_model=d_test, num_heads=heads_test, dropout=0.0).eval()
gqa_layer = GQAAttention(d_model=d_test, num_q_heads=heads_test, num_kv_heads=heads_test).eval()

with torch.no_grad():
    gqa_layer.w_q.weight.copy_(mha_layer.w_q.weight)
    gqa_layer.w_k.weight.copy_(mha_layer.w_k.weight)
    gqa_layer.w_v.weight.copy_(mha_layer.w_v.weight)
    gqa_layer.w_o.weight.copy_(mha_layer.w_o.weight)

test_input = torch.randn(2, 5, d_test)
out_mha, _ = mha_layer(test_input, causal=True)
out_gqa = gqa_layer(test_input)

torch.testing.assert_close(out_mha, out_gqa, atol=1e-6, rtol=1e-5)
print("[PASS] Cross-module verification: Part 1 MHA matches Part 3 GQA (when H_Q == H_KV) within 1e-6 tolerance!")


Loaded GQAAttention and KVCache from Part 3.
[PASS] Cross-module verification: Part 1 MHA matches Part 3 GQA (when H_Q == H_KV) within 1e-6 tolerance!


## 7. Systems Architect Audit & Review Summary

- **Core Attention Implementation:** Pure PyTorch without high-level library wrappers.
- **Variance Preservation:** $\frac{1}{\sqrt{d_k}}$ scaling formally verified and empirically validated ($\text{Var} \approx 1.0027$).
- **Causal Masking & Padding:** Upper-triangular elements zeroed; padding tokens masked; zero gradient leakage to padding positions.
- **Precision:** Float32 score accumulation ensures complete stability in FP16 / BF16.
- **Cross-Station Compatibility:** Formally validated against Data & Payload Specialist's Part 3 GQA implementation.
- **Squad Sign-Off:** Ready for final mission transmission.
